# Step5-1 모델B (ROI 제외) 분리 노트북

- 목적: LTV/CAC를 제외한 모델B 점수로 채널 우선순위를 산출
- 가중치: CVR 50%, ARPU 25%, Retention(D7+D30) 25%



In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# 데이터 로드
start_dt = pd.Timestamp('2025-03-01')
end_dt = pd.Timestamp('2025-05-31')
users = pd.read_csv('data/users.csv', parse_dates=['signup_date'])
payments = pd.read_csv('data/payment_transactions.csv', parse_dates=['transaction_date'])
events = pd.read_csv('data/event_logs.csv', parse_dates=['event_timestamp'])

cohort = users[(users['signup_date'] >= start_dt) & (users['signup_date'] <= end_dt)].copy()
cohort_ids = cohort['user_id'].unique()

# CVR + ARPU (completed 결제 기준)
paid = payments[(payments['user_id'].isin(cohort_ids)) & (payments['status'] == 'completed')]
revenue = paid.groupby('user_id')['amount'].sum().reset_index()
merged = pd.merge(cohort[['user_id', 'acquisition_source', 'signup_date']], revenue, on='user_id', how='left').fillna(0)
merged['is_paid'] = merged['amount'] > 0

ch = merged.groupby('acquisition_source').agg(
    user_count=('user_id', 'count'),
    paid_count=('is_paid', 'sum'),
    cvr=('is_paid', 'mean'),
    arpu=('amount', 'mean')
).reset_index()

# Retention(D7 + D30)
ev = events[events['user_id'].isin(cohort_ids)].merge(cohort[['user_id', 'signup_date', 'acquisition_source']], on='user_id', how='inner')
ev['gap_days'] = (ev['event_timestamp'].dt.normalize() - ev['signup_date'].dt.normalize()).dt.days

d7_ids = ev[(ev['gap_days'] >= 1) & (ev['gap_days'] <= 7)]['user_id'].unique()
d30_ids = ev[(ev['gap_days'] >= 15) & (ev['gap_days'] <= 30)]['user_id'].unique()
merged['d7'] = merged['user_id'].isin(d7_ids)
merged['d30'] = merged['user_id'].isin(d30_ids)
ret = merged.groupby('acquisition_source').agg(d7_rate=('d7', 'mean'), d30_rate=('d30', 'mean')).reset_index()

df = ch.merge(ret, on='acquisition_source', how='left').fillna(0)

def norm(s):
    if s.max() == s.min():
        return pd.Series([100] * len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

def grade(x):
    if x >= 85: return 'S'
    if x >= 70: return 'A'
    if x >= 55: return 'B'
    if x >= 40: return 'C'
    return 'D'

# 모델B 점수
df['s_cvr'] = norm(df['cvr'])
df['s_arpu'] = norm(df['arpu'])
df['s_ret'] = norm(df['d7_rate'] + df['d30_rate'])
df['final_score_no_roi'] = df['s_cvr'] * 0.50 + df['s_arpu'] * 0.25 + df['s_ret'] * 0.25

df = df.sort_values('final_score_no_roi', ascending=False).reset_index(drop=True)
df['grade'] = df['final_score_no_roi'].apply(grade)

print('=== 모델B 최종 종합 스코어 (ROI 제외) ===')
display(df[['acquisition_source','cvr','arpu','d7_rate','d30_rate','final_score_no_roi','grade']]
        .style
        .format({'cvr':'{:.1%}','arpu':'{:,.1f}','d7_rate':'{:.1%}','d30_rate':'{:.1%}','final_score_no_roi':'{:.1f}'})
        .highlight_max(subset=['final_score_no_roi'], color='#d9ead3'))

print('모델B 상위 3채널:', ', '.join(df.head(3)['acquisition_source']))

